In [2]:
!pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, datediff, current_date
spark = SparkSession.builder \
    .appName("SupplyChainProcessing") \
    .getOrCreate()


In [5]:
from google.colab import files
uploaded = files.upload()

Saving orders.csv to orders.csv


In [6]:
df = spark.read.csv("orders.csv", header=True, inferSchema=True)
df.show()

+--------+-----------+----------+--------+-------------+---------+
|order_id|supplier_id|      item|quantity|delivery_date|   status|
+--------+-----------+----------+--------+-------------+---------+
|       1|         S1|    Laptop|      10|   2024-01-01|Delivered|
|       2|         S2|     Mouse|      50|   2024-01-15|  Pending|
|       3|         S1|  Keyboard|      30|   2024-01-10|  Delayed|
|       4|         S3|   Monitor|       5|   2023-12-20|Delivered|
|       5|         S2|    Webcam|      20|   2024-02-01|  Pending|
|       6|         S1|Headphones|      15|   2023-11-15|  Delayed|
|       7|         S3|   Charger|      40|   2023-12-01|  Delayed|
|       8|         S2|   USB Hub|      25|   2024-01-20|  Pending|
|       9|         S1|       SSD|       8|   2023-10-10|  Delayed|
|      10|         S3|   Printer|       3|   2024-02-10|  Pending|
+--------+-----------+----------+--------+-------------+---------+



In [7]:
df.printSchema()


root
 |-- order_id: integer (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- item: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- delivery_date: date (nullable = true)
 |-- status: string (nullable = true)



In [8]:
# Convert delivery_date to DateType if needed
from pyspark.sql.functions import to_date
df = df.withColumn("delivery_date", to_date(col("delivery_date"), "yyyy-MM-dd"))

In [9]:
# Calculate delay days
df = df.withColumn("delay_days", datediff(current_date(), col("delivery_date")))

In [10]:
# Flag delayed shipments (delay > 0 days)
df = df.withColumn("is_delayed", when(col("delay_days") > 0, 1).otherwise(0))

In [11]:
# Filter Delayed Shipments
delayed_df = df.filter(col("is_delayed") == 1)
print("Delayed Shipments:")
delayed_df.show()

Delayed Shipments:
+--------+-----------+----------+--------+-------------+---------+----------+----------+
|order_id|supplier_id|      item|quantity|delivery_date|   status|delay_days|is_delayed|
+--------+-----------+----------+--------+-------------+---------+----------+----------+
|       1|         S1|    Laptop|      10|   2024-01-01|Delivered|       863|         1|
|       2|         S2|     Mouse|      50|   2024-01-15|  Pending|       849|         1|
|       3|         S1|  Keyboard|      30|   2024-01-10|  Delayed|       854|         1|
|       4|         S3|   Monitor|       5|   2023-12-20|Delivered|       875|         1|
|       5|         S2|    Webcam|      20|   2024-02-01|  Pending|       832|         1|
|       6|         S1|Headphones|      15|   2023-11-15|  Delayed|       910|         1|
|       7|         S3|   Charger|      40|   2023-12-01|  Delayed|       894|         1|
|       8|         S2|   USB Hub|      25|   2024-01-20|  Pending|       844|         1|
| 

In [12]:
# Group by Supplier and Count Delayed Orders
grouped_df = delayed_df.groupBy("supplier_id") \
    .count() \
    .withColumnRenamed("count", "delayed_order_count") \
    .orderBy(col("delayed_order_count").desc())


In [13]:
print("Delayed Orders by Supplier:")
grouped_df.show()

Delayed Orders by Supplier:
+-----------+-------------------+
|supplier_id|delayed_order_count|
+-----------+-------------------+
|         S1|                  4|
|         S3|                  3|
|         S2|                  3|
+-----------+-------------------+



In [14]:
# Save to CSV
grouped_df.write.mode("overwrite").csv("output/delayed_by_supplier_csv")


In [15]:
# Save to Parquet
grouped_df.write.mode("overwrite").parquet("output/delayed_by_supplier_parquet")


In [18]:
print("Output saved to CSV and Parquet.")

spark.stop()

Output saved to CSV and Parquet.
